In [1]:
"""
====================================================================
 Descarga de datos climáticos históricos de Ushuaia (Open-Meteo)
====================================================================
Proyecto : Predicción de Demanda Turística — Ushuaia
Autor    : Darío Martínez
Fuente   : Open-Meteo Historical Weather API (reanálisis ERA5)

Qué hace este script:
  1. Consulta la API histórica de Open-Meteo para las coordenadas de
     Ushuaia, en el período 2004-2025, pidiendo datos DIARIOS.
  2. Agrega esos datos a frecuencia MENSUAL:
        - temperatura  -> promedio del mes
        - precipitación -> suma del mes
        - viento máximo -> máximo del mes
  3. Exporta el resultado a un archivo Excel (.xlsx).

Requisitos:  pip install requests pandas openpyxl
====================================================================
"""

import requests
import pandas as pd

# ── 1. Parámetros de la consulta ────────────────────────────────────
LAT_USHUAIA = -54.8019      # latitud de Ushuaia
LON_USHUAIA = -68.3030      # longitud de Ushuaia
FECHA_INICIO = "2004-01-01"
FECHA_FIN    = "2025-11-30"

URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude":   LAT_USHUAIA,
    "longitude":  LON_USHUAIA,
    "start_date": FECHA_INICIO,
    "end_date":   FECHA_FIN,
    # Variables DIARIAS que luego se agregan a mensual
    "daily": [
        "temperature_2m_mean",   # temperatura media diaria
        "precipitation_sum",     # precipitación total diaria
        "wind_speed_10m_max",    # viento máximo diario
    ],
    "timezone": "America/Argentina/Ushuaia",
}

# ── 2. Llamada a la API ─────────────────────────────────────────────
print("Descargando datos de Open-Meteo (ERA5) para Ushuaia...")
resp = requests.get(URL, params=params, timeout=60)
resp.raise_for_status()          # corta si hubo un error HTTP
data = resp.json()

# ── 3. Pasar la respuesta JSON a un DataFrame diario ────────────────
daily = pd.DataFrame({
    "fecha":               pd.to_datetime(data["daily"]["time"]),
    "temperature_2m_mean": data["daily"]["temperature_2m_mean"],
    "precipitation_sum":   data["daily"]["precipitation_sum"],
    "wind_speed_10m_max":  data["daily"]["wind_speed_10m_max"],
})
print(f"  Registros diarios descargados: {len(daily):,}")

# ── 4. Agregación a frecuencia MENSUAL ──────────────────────────────
#   temperatura  -> promedio   |  lluvia -> suma   |  viento -> máximo
daily["anio"] = daily["fecha"].dt.year
daily["mes"]  = daily["fecha"].dt.month

mensual = (
    daily.groupby(["anio", "mes"])
         .agg(temperature_2m_mean=("temperature_2m_mean", "mean"),
              precipitation_sum   =("precipitation_sum",   "sum"),
              wind_speed_10m_max  =("wind_speed_10m_max",  "max"))
         .reset_index()
)

# Columna de fecha al primer día de cada mes (para integrar con la serie turística)
mensual["fecha"] = pd.to_datetime(
    mensual["anio"].astype(str) + "-" + mensual["mes"].astype(str) + "-01")

# Redondeo prolijo
mensual["temperature_2m_mean"] = mensual["temperature_2m_mean"].round(2)
mensual["precipitation_sum"]   = mensual["precipitation_sum"].round(1)
mensual["wind_speed_10m_max"]  = mensual["wind_speed_10m_max"].round(1)

# Reordenar columnas
mensual = mensual[["fecha", "anio", "mes",
                   "temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max"]]

print(f"  Registros mensuales generados: {len(mensual):,}")

# ── 5. Exportar a Excel ─────────────────────────────────────────────
ARCHIVO_SALIDA = "clima_ushuaia_mensual_2004_2025.xlsx"
mensual.to_excel(ARCHIVO_SALIDA, index=False, sheet_name="clima_mensual")

print(f"\n✓ Archivo generado: {ARCHIVO_SALIDA}")
print("\nVista previa:")
print(mensual.head(12).to_string(index=False))

Descargando datos de Open-Meteo (ERA5) para Ushuaia...
  Registros diarios descargados: 8,005
  Registros mensuales generados: 263

✓ Archivo generado: clima_ushuaia_mensual_2004_2025.xlsx

Vista previa:
     fecha  anio  mes  temperature_2m_mean  precipitation_sum  wind_speed_10m_max
2004-01-01  2004    1                 9.83              112.9                30.3
2004-02-01  2004    2                11.49               62.0                22.9
2004-03-01  2004    3                 8.04               58.1                23.7
2004-04-01  2004    4                 5.20               53.1                21.8
2004-05-01  2004    5                 1.57               49.1                18.8
2004-06-01  2004    6                 0.28               77.5                13.8
2004-07-01  2004    7                -1.93               35.9                19.5
2004-08-01  2004    8                -0.26               30.6                21.0
2004-09-01  2004    9                 0.89               6